# Kolmogorov Flow 訓練與評估（Colab + uv）

本筆記本包含：
- 掛載 Google Drive
- 安裝依賴（含 SOAP）
- 使用 uv run 進行訓練
- 以 checkpoint 進行評估


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TODO: 更新成你的 repo 位置
repo_path = '/content/drive/MyDrive/jaxpi'
%cd {repo_path}

In [ ]:
# 系統依賴
!apt-get -y update
!apt-get -y install git

In [ ]:
# 安裝 uv
!python3 -m pip install -U uv

In [ ]:
# Python 依賴（Colab）
# 如果需要 SOAP，會從 GitHub 取得
!uv pip install git+https://github.com/haydn-jones/SOAP_JAX.git
# 安裝本專案（必要時）
!uv pip install -e .

In [ ]:
# 訓練參數
config_path = 'examples/kolmogorov_flow/configs/soap.py'  # 或 pirate.py
workdir = f'{repo_path}/runs/kf_soap_colab'

!uv run python examples/kolmogorov_flow/main.py \
  --config={config_path} \
  --workdir={workdir}

## 評估（Checkpoint）

以下會自動尋找包含 `time_window_` 的 checkpoint 目錄。


In [ ]:
from pathlib import Path

def find_checkpoint_root(root):
    root = Path(root)
    candidates = []
    for path in root.rglob('time_window_1'):
        candidates.append(path.parent)
    return candidates

checkpoint_roots = find_checkpoint_root(workdir)
print('找到的 checkpoint 根目錄:')
for p in checkpoint_roots:
    print('-', p)

# TODO: 選擇正確的 checkpoint 根目錄
checkpoint_path = str(checkpoint_roots[0]) if checkpoint_roots else ''
print('checkpoint_path =', checkpoint_path)

In [ ]:
# 使用完整時間窗評估（較耗記憶體）
output_path = f'{workdir}/eval_soap_full.npz'

!uv run python examples/kolmogorov_flow/evaluate_checkpoint.py \
  --config soap \
  --checkpoint_path={checkpoint_path} \
  --output={output_path}

In [ ]:
# 若遇到 OOM，改用簡化評估（每個窗口最後一步）
output_path = f'{workdir}/eval_soap_final_step.npz'

!uv run python examples/kolmogorov_flow/evaluate_checkpoint.py \
  --config soap \
  --checkpoint_path={checkpoint_path} \
  --mode final_step \
  --device cpu \
  --output={output_path}